# Tool-Router Ladder: Stage E1, data-scaling ablation (1.5B SFT at 25% and 50%)

Same pipeline and hyperparameters as the v1 1.5B adapter, on a seeded **task-id** subset of the
teacher data (`src/subsets.py`): 12 of 46 tasks at 25%, 23 at 50%, nested, every episode of a chosen
task kept. The 100% point is the v1 adapter, so it is not retrained.

**Settings:** Accelerator **GPU T4 x2** (one card is enough at 1.5B), Internet **on**, the same
tool-router SFT dataset attached as for v1 training (the one with `manifest.json`).
**Secrets:** `HF_TOKEN` (write) and `WANDB_API_KEY`, both ticked.
**Run:** Save Version -> **Save & Run All (Commit)**. Checkpoints go to the Hub every epoch and a
rerun resumes. **No API quota.**

**Expected:** v1 1.5B took 51 min on the full set, so roughly 15 min (25%) and 30 min (50%) of training,
plus model loading. **Do not install vLLM here.**

**Send back:** the two `[subsets]` lines and the two `[train] done: ...` lines.

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN", "WANDB_API_KEY"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN", "WANDB_API_KEY"])

In [ ]:
# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

!pip install -q peft
# Kaggle ships torchao 0.10.0; peft's LoRA dispatcher raises on anything below
# 0.16.0 instead of skipping it. Nothing here uses torchao, and with the package
# gone peft's is_torchao_available() returns False and dispatch falls through.
!pip uninstall -y -q torchao

In [ ]:
# Finds the attached dataset wherever Kaggle mounted it (it must contain manifest.json).
import glob, os
hits = glob.glob("/kaggle/input/**/manifest.json", recursive=True)
assert hits, "attach the tool-router dataset (Add Input) - no manifest.json under /kaggle/input"
DATA = os.path.dirname(hits[0])
HF_USER = "madhusiddharths1"   # your Hugging Face username
print("DATA =", DATA); print(sorted(os.listdir(DATA)))

### 1. Build the subsets (seconds; the manifests keep every key train.py checks)

In [ ]:
for f, tag in [(0.25, "f25"), (0.5, "f50")]:
    !python src/subsets.py --config configs/v2/qwen15b-e1.yaml --data "$DATA" --fraction {f} --out /kaggle/working/sft-{tag}

### 2. 25% of the teacher tasks

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train.py --config configs/v2/qwen15b-e1.yaml --data /kaggle/working/sft-f25 --output-dir /kaggle/working/qwen15b-f25 --hub-repo "$HF_USER/tool-router-qwen15b-sft-f25" --run-name qwen15b-sft-f25 --wandb

### 3. 50% of the teacher tasks

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python src/train.py --config configs/v2/qwen15b-e1.yaml --data /kaggle/working/sft-f50 --output-dir /kaggle/working/qwen15b-f50 --hub-repo "$HF_USER/tool-router-qwen15b-sft-f50" --run-name qwen15b-sft-f50 --wandb